# Brain Tumor Classification — Custom CNN

Task 2: Build a custom CNN from scratch targeting ≥ 98% accuracy (Challenge 1).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print('TensorFlow version:', tf.__version__)

## 1. Configuration

In [ ]:
DATA_DIR = '../data/brain_tumor_dataset'
TRAIN_DIR = os.path.join(DATA_DIR, 'Training')
TEST_DIR  = os.path.join(DATA_DIR, 'Testing')

IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 50
CLASSES     = ['glioma', 'meningioma', 'no_tumor', 'pituitary']
NUM_CLASSES = len(CLASSES)
MODEL_SAVE_PATH = '../models/custom_cnn_brain_tumor.keras'

## 2. Data Pipeline

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.15,
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=True, seed=42,
)
val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', shuffle=False, seed=42,
)
test_gen = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False,
)
print('Class indices:', train_gen.class_indices)

## 3. Custom CNN Architecture

### Design strategy

The network is built from three components:

- **Stem** — two lightweight conv blocks extract low-level features (edges, textures) before any spatial downsampling.
- **Residual stages 1–4** — filter depth doubles each stage (128 → 256 → 512) to learn progressively abstract features. Residual shortcuts improve gradient flow and allow deeper training without degradation.
- **Classifier head** — `GlobalAveragePooling2D` replaces `Flatten` to dramatically reduce parameters and act as a built-in spatial regulariser; two dense layers then map the pooled feature vector to class probabilities.

### Regularisation rationale

| Location | Rate | Rationale |
|---|---|---|
| Conv blocks | BN (no bias) | `use_bias=False` in Conv2D is correct because BatchNorm's β parameter absorbs the offset, making a separate bias redundant |
| Stem + Stages 1–2 | Dropout 0.25 | Light dropout on spatial features — too-aggressive dropout here destroys spatially-structured information |
| Stage 3 | Dropout 0.30 | Nudged up at peak capacity (512 filters), where the model is most prone to overfitting |
| Stage 4 | None before GAP | `GlobalAveragePooling2D` already averages over the full spatial map — adding dropout before it would be double-regularising |
| Dense(512) | Dropout 0.50 | Dense layers have no spatial redundancy; 0.5 is the standard strong regulariser for the widest fully-connected layer |
| Dense(256) | Dropout 0.30 | Lighter after the representation is already compressed to 256 units — 0.5 here risks underfitting |

In [ ]:
def conv_block(x, filters, kernel_size=3, strides=1):
    # use_bias=False: BN's beta absorbs the offset, making a conv bias redundant.
    x = layers.Conv2D(filters, kernel_size, strides=strides, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x


def residual_block(x, filters):
    shortcut = x
    x = conv_block(x, filters)
    x = conv_block(x, filters)
    # Project the shortcut to the new channel depth only when it changes;
    # BN on the projection keeps the shortcut on the same scale as the main path.
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x


def build_custom_cnn(num_classes: int, image_size: tuple) -> keras.Model:
    inputs = keras.Input(shape=(*image_size, 3))

    # Stem: capture low-level features before the first downsampling step.
    # Two conv blocks keep the receptive field small so the network sees fine
    # MRI texture before pooling compresses it.
    x = conv_block(inputs, 32, kernel_size=3)
    x = conv_block(x, 64, kernel_size=3)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)        # light — spatial features have natural channel redundancy

    # Stage 1 — 128 filters
    x = residual_block(x, 128)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)

    # Stage 2 — 256 filters
    x = residual_block(x, 256)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)

    # Stage 3 — 512 filters; nudge dropout up because at peak capacity the
    # model is most at risk of memorising training examples.
    x = residual_block(x, 512)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.3)(x)         # slightly higher at max filter depth

    # Stage 4 — no dropout before GAP; GlobalAveragePooling2D already averages
    # over the entire spatial map, acting as a strong spatial regulariser.
    x = residual_block(x, 512)
    x = layers.GlobalAveragePooling2D()(x)

    # Classifier head — dense layers have no spatial redundancy, so they
    # overfit faster; hit the widest layer hard, ease off as width shrinks.
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)         # 0.5: standard aggressive regularisation for the widest FC layer
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)         # 0.3: lighter after compression — 0.5 here would underfit
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs)


model = build_custom_cnn(NUM_CLASSES, IMAGE_SIZE)
model.summary()

## 4. Train

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=4, min_lr=1e-7, verbose=1),
    keras.callbacks.ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True, verbose=1),
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## 5. Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(test_gen)
print(f'Test accuracy: {test_acc:.4f}')

y_pred = np.argmax(model.predict(test_gen), axis=1)
y_true = test_gen.classes

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASSES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Greens')
plt.title('Confusion Matrix — Custom CNN')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout(); plt.show()

## 6. Saliency Map Visualization

In [ ]:
import tensorflow as tf

def compute_saliency(model, image: np.ndarray) -> np.ndarray:
    img_tensor = tf.Variable(image[np.newaxis, ...], dtype=tf.float32)
    with tf.GradientTape() as tape:
        preds = model(img_tensor)
        top_class = tf.argmax(preds[0])
        loss = preds[:, top_class]
    grads = tape.gradient(loss, img_tensor)
    saliency = tf.reduce_max(tf.abs(grads), axis=-1)[0].numpy()
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
    return saliency


images, labels = next(test_gen)
sample = images[0]
saliency = compute_saliency(model, sample)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(sample); axes[0].set_title('Original MRI'); axes[0].axis('off')
axes[1].imshow(saliency, cmap='hot'); axes[1].set_title('Saliency Map'); axes[1].axis('off')
overlay = sample.copy()
overlay[:, :, 0] = np.clip(overlay[:, :, 0] + 0.5 * saliency, 0, 1)
axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
plt.suptitle(f'Predicted: {CLASSES[np.argmax(model.predict(sample[np.newaxis, ...]))]}  |  True: {CLASSES[np.argmax(labels[0])]}')
plt.tight_layout(); plt.show()